In [9]:
import pandas as pd

# Load the Excel file
df = pd.read_excel("Thematic_Analysis_Final.xlsx")

# Metadata columns
metadata_cols = [
    'S2 - Topic Disussed', 'Topic Quote', 'Topic Important',
    'Quote Number', 'Participant Number', 'Category - L1', 'Sub Category - L2'
]

# One-hot columns start from column 8 (index 7)
one_hot_cols = df.columns[7:]

# --- View shape ---
print("DataFrame shape:", df.shape)

# --- View head ---
print("\nDataFrame head:")
print(df.head())

# --- Duplicate check for Topic Quote ---
duplicate_quotes = df[df['Topic Quote'].duplicated(keep=False)]

print("\nNumber of duplicate Topic Quotes:", duplicate_quotes.shape[0])
print("\nDuplicate Quotes:")
print(duplicate_quotes[['Topic Quote', 'Quote Number', 'Participant Number']])



DataFrame shape: (186, 30)

DataFrame head:
                                         Topic Quote  Quote Number  \
0  But for a higher number, definitely we need a ...            43   
1  But in decision making - clearly, I think it h...            44   
2  For exceptions that a manual underwrite would ...            45   
3  I feel like if AI is not a human being AI can'...            46   
4  See, the underwriting process is a very comple...            47   

   Participant Number Gender Location  Years of Experience  \
0                 5.0      M   Europe                 12.0   
1                 1.0      M   Europe                  4.0   
2                 4.0      F       US                 20.0   
3                 2.0      F       US                 15.0   
4                 8.0      M    India                 10.0   

             Category - L1  Manual Intervention in Complex or High-Risk Cases  \
0  Human Oversight Control                                                1.0   


In [10]:
# Remove exact duplicate quotes (case-sensitive) and overwrite df
df = df.drop_duplicates(subset=['Topic Quote'], keep='first').reset_index(drop=True)

print("Rows after removing duplicates:", df.shape[0])


Rows after removing duplicates: 186


In [11]:
# Basic stats
num_quotes = df.shape[0]
num_participants = df['Participant Number'].nunique()

# Frequency of main categories
category_counts = df['Category - L1'].value_counts()

# --- FIX: convert one-hot columns to numeric ---
df[one_hot_cols] = df[one_hot_cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)

# Count of 1s in one-hot columns
one_hot_counts = df[one_hot_cols].sum().sort_values(ascending=False)

# Quotes per participant
quotes_per_participant = df.groupby('Participant Number')['Quote Number'].count()

# Display results
print("Total quotes:", num_quotes)
print("Total participants:", num_participants)

print("\nCategory counts:\n", category_counts)

print("\nOne-hot column counts:\n", one_hot_counts)

print("\nQuotes per participant:\n", quotes_per_participant)



Total quotes: 186
Total participants: 10

Category counts:
 Category - L1
Risks & Concerns           31
Semi Factual               25
Factual Explaination       23
Actionability              22
Counterfactual             22
Technical Elements         20
Explanation Format         19
Human Oversight Control    14
Regulatory & Complaince    10
Name: count, dtype: int64

One-hot column counts:
 Clarity and Specificity                               33
Validation and Justification of Decisions             32
Usefulness and Efficiency                             32
Data Integrity and Security Concerns                  16
Collaboration between AI and Humans                   16
Emphasis on Individualized Decision Making            14
Scenario based explanations                           10
Preference for Combined Text and Visuals              10
Human Judgment, Communication, and Contextual Gaps     9
Support in Complex or Borderline Case                  9
Other                              

In [12]:
import pandas as pd

# Assuming df and one_hot_cols are already defined

# Convert one-hot columns to numeric if not already done
df[one_hot_cols] = df[one_hot_cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)

# Loop through each Category - L1
for category in df['Category - L1'].unique():
    # Filter rows for the category
    cat_df = df[df['Category - L1'] == category]
    
    # Sum one-hot columns for this category
    cat_one_hot_sum = cat_df[one_hot_cols].sum()
    
    # Keep only columns where sum > 0
    cat_one_hot_sum = cat_one_hot_sum[cat_one_hot_sum > 0]
    
    # Skip if no columns have values
    if cat_one_hot_sum.empty:
        continue
    
    # Convert to DataFrame
    cat_table = pd.DataFrame(cat_one_hot_sum).rename(columns={0: 'Count'})
    
    # Add total row
    total_row = pd.DataFrame(cat_table.sum()).T
    total_row.index = ['Total']
    cat_table = pd.concat([cat_table, total_row])
    
    # Display table
    print(f"\nCategory: {category}")
    display(cat_table)





Category: Human Oversight Control


,Count
Manual Intervention in Complex or High-Risk Cases,6
Collaboration between AI and Humans,8
Total,14



Category: Risks & Concerns


,Count
"Model Accuracy, Validation, and Comprehensiveness""",4
"Human Judgment, Communication, and Contextual Gaps",9
Data Integrity and Security Concerns,16
Other,4
Total,33



Category: Technical Elements


...

,Count
Professional Confidence & Judgement,1
Validation and Justification of Decisions,13
Clarity and Specificity,7
Emphasis on Individualized Decision Making,4
Usefulness and Efficiency,16
Support in Complex or Borderline Case,7
Total,48



Category: Explanation Format


,Count
Preference for Combined Text and Visuals,10
Graphical Clarity and Ease of Interpretation,3
Textual Detail for Depth and Context,4
Interactive and Feedback-Enabled Systems,5
Total,22



Category: Actionability


,Count
Collaboration between AI and Humans,8
Validation and Justification of Decisions,10
Usefulness and Efficiency,2
Scenario based explanations,10
Total,30


In [13]:
import pandas as pd

# Ensure one-hot columns are numeric
df[one_hot_cols] = df[one_hot_cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)

# Loop through each Category - L1
for category in df['Category - L1'].unique():
    # Filter rows for the category
    cat_df = df[df['Category - L1'] == category].copy()
    
    # Keep only one-hot columns with any 1s in this category
    cat_one_hot_cols = cat_df[one_hot_cols].sum()
    cat_one_hot_cols = cat_one_hot_cols[cat_one_hot_cols > 0].index.tolist()
    
    if not cat_one_hot_cols:  # skip if no one-hot columns have values
        continue
    
    # Select relevant columns
    cat_table = cat_df[['Topic Quote', 'Quote Number', 'Participant Number'] + cat_one_hot_cols].copy()
    
    # Convert Quote Number and Participant Number to nullable integers
    cat_table['Quote Number'] = cat_table['Quote Number'].astype('Int64')
    cat_table['Participant Number'] = cat_table['Participant Number'].astype('Int64')
    
    # Add totals row for one-hot columns
    totals = pd.DataFrame(cat_table[cat_one_hot_cols].sum()).T
    totals['Topic Quote'] = 'Total'
    totals['Quote Number'] = pd.NA
    totals['Participant Number'] = pd.NA
    
    # Reorder columns to match
    totals = totals[cat_table.columns]
    
    # Append totals row
    cat_table = pd.concat([cat_table, totals], ignore_index=True)
    
    # Display the table with formatting:
    # - Topic Quote: left-aligned
    # - Numbers and one-hot columns: right-aligned
    print(f"\nCategory: {category}")
    display(
        cat_table.style.set_properties(subset=['Topic Quote'], **{'text-align': 'left'})
                  .set_properties(subset=cat_table.columns[1:], **{'text-align': 'right'})
                  .hide(axis="index")
    )



Category: Human Oversight Control


/tmp/ipykernel_31417/3480308812.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cat_table = pd.concat([cat_table, totals], ignore_index=True)


Topic Quote,Quote Number,Participant Number,Manual Intervention in Complex or High-Risk Cases,Collaboration between AI and Humans
"But for a higher number, definitely we need a manual intervention",43,5,1,0
"But in decision making - clearly, I think it has to go hand in hand with the human",44,1,0,1
For exceptions that a manual underwrite would need to be done,45,4,1,0
"I feel like if AI is not a human being AI can't put himself in, in the daily day-to-day affairs of human beings",46,2,0,1
"See, the underwriting process is a very complex process, and the whole thing depends upon the loan segment so we will think differently if it is a personal segment loan like housing, loan, mortgage, loan, or vehicle loan. So those case will be different.",47,8,1,0
"But when coming to the maximum loan requirement on a property or on a, on a security, you know, there are chances that AI might take a, a bad decision on an application where we don't know whether the applicant is having the ability to repay or the other scenarios behind it. But when the risk is high, definitely there should be some kind of manual intervention to either approve or decline",48,3,1,0
I do feel like underwriting does always need some sort of human element to it,49,6,0,1
"while this is not auto approved in this cases, it can be approved by, say, an underwriter subject to 1, 2, 3 being met",50,7,1,0
"But still we need, you know, like human eye to verify",51,9,0,1
"We need to have both the perspective of AI and human intelligence also, So AI need identify the fraud and the loopholes and we need to have the human intelligence",52,10,0,1



Category: Risks & Concerns


/tmp/ipykernel_31417/3480308812.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cat_table = pd.concat([cat_table, totals], ignore_index=True)


...

/tmp/ipykernel_31417/3480308812.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cat_table = pd.concat([cat_table, totals], ignore_index=True)


Topic Quote,Quote Number,Participant Number,Preference for Combined Text and Visuals,Graphical Clarity and Ease of Interpretation,Textual Detail for Depth and Context,Interactive and Feedback-Enabled Systems
"These are vsually pleasing, easy to look at, not too overwhelming.",149,2,1,0,0,0
"I think, both text and graphs because I can tell you not everybody is good at reading graphs, and some people love it",150,7,1,0,0,0
I think the factual graph is easier to read than the other,151,7,0,1,0,0
On the factual explanations-graphs in this model is a graph is the best.,152,6,0,1,0,0
For the semi factual - both text and graph are needed,153,6,1,0,0,0
"I think it should be both because sometimes the the sanctioning authority, they might not have the much of time to go through the text so they can just visualize and thing.",154,8,1,0,0,0
Both. - text and visuals are needed,155,4,1,0,0,0
The bar graph -I feel like is the best is me looking at it. It's easy to understand.,156,1,0,1,0,0
"For the factual explanation - ""both"" and the other "" I would just say the text is best""",157,3,1,0,0,0
"Both will be good, because, you know, like some people does not understand graph, but some people do.",158,10,1,0,0,0



Category: Actionability


/tmp/ipykernel_31417/3480308812.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cat_table = pd.concat([cat_table, totals], ignore_index=True)


Topic Quote,Quote Number,Participant Number,Collaboration between AI and Humans,Validation and Justification of Decisions,Usefulness and Efficiency,Scenario based explanations
"The actions (based off the explanations) would definitely be the finalization of whether or not the loan gets approved as far as with those graphs that I reviewed along with my own observation, as far as the documentation I have in front of me. This actually helps to streamline the process",166,4,0,1,0,0
"another part of when we underwrite is, we need to explain why we're doing what we're doing. And that basically helps explain, candace: Or sometimes we also need compensating factors for things. You know for why we are approving well this just told us",167,2,0,1,0,0
So that will be a very useful for the decision makers who will be the ultimate authority to approve the loans,168,8,1,1,0,0
"Yes this would help - ""The the underwriters which we were doing right now, it's manual manual underwriting.""",169,6,1,1,0,0
making a decision based on these data sets whatever you have shown me - we need to deep dive into the into the more data or more analysis.,170,8,1,1,0,0
"the actions would definitely be the finalization of whether or not the loan gets approved as far as with those graphs that I reviewed along with my own observation,",171,4,1,1,0,0
"Not absolutely letting decision making to AI, but the integration with humans, it would actually would actually be superb.",172,1,1,1,0,0
I will find that useful in my in determining an approval or decline - but still I have to look at it.,173,3,1,1,0,0
"The action is like, it's our call. But the explanations would help in making the decision better.",174,10,1,0,0,0
"For me to come to that particular explanation, it would take 1520 min for me to read out those documents, but it is already been done by an AI and giving me to you don't have to look all those, and just look only this particular point, and which is reducing my work.",175,5,1,1,0,0


In [14]:
import pandas as pd

# Assuming your final DataFrame is called df
output_file = "cleaned_underwriter_final.xlsx"

# Export to Excel
df.to_excel(output_file, index=False)

print(f"Data exported successfully to {output_file}")


Data exported successfully to cleaned_underwriter_final.xlsx
